# 05 — Ingénierie des variables

**Objectif** : produire `data/processed/features.csv`.

Cible : le nombre de ventes du mois suivant, pour une région et une plage de
prix données. Toutes les variables explicatives sont calculées à partir
d'informations disponibles **avant** le mois prédit.

In [1]:
import importlib
import sys
from pathlib import Path

racine = Path.cwd().resolve()
if not (racine / "src").is_dir():
    racine = racine.parent
sys.path.insert(0, str(racine))

import src

importlib.reload(src)


import numpy as np
import pandas as pd

from src import FEATURES_CSV, VENTES_PROPRES_CSV

pd.set_option("display.max_columns", None)
CLES_SERIE = ["id_region", "code_plage_prix"]

In [2]:
df = pd.read_csv(
    VENTES_PROPRES_CSV,
    parse_dates=["periode"],
    dtype={"id_region": str, "code_plage_prix": str, "code_nature_acte": str},
)
df = df.sort_values([*CLES_SERIE, "periode"]).reset_index(drop=True)
df.head()

,periode,id_region,code_plage_prix,code_nature_acte,nombre_ventes,region,plage_prix,nature_acte
0,2021-01-01,01,1,001,240,Bas-Saint-Laurent,Moins de 250 000 $,Vente
1,2021-02-01,01,1,001,327,Bas-Saint-Laurent,Moins de 250 000 $,Vente
2,2021-03-01,01,1,001,419,Bas-Saint-Laurent,Moins de 250 000 $,Vente
3,2021-04-01,01,1,001,486,Bas-Saint-Laurent,Moins de 250 000 $,Vente
4,2021-05-01,01,1,001,566,Bas-Saint-Laurent,Moins de 250 000 $,Vente


## Variables calendaires

Le mois est encodé en sinus/cosinus pour que décembre et janvier soient proches.

In [3]:
df["annee"] = df["periode"].dt.year
df["mois"] = df["periode"].dt.month
df["trimestre"] = df["periode"].dt.quarter
df["mois_sin"] = np.sin(2 * np.pi * df["mois"] / 12)
df["mois_cos"] = np.cos(2 * np.pi * df["mois"] / 12)

## Retards et moyennes mobiles

Les retards sont calculés **par série** (région × plage). `shift` avant
`rolling` garantit qu'aucune fenêtre n'inclut le mois courant.

In [4]:
groupes = df.groupby(CLES_SERIE, group_keys=False)

for retard in (1, 2, 3, 12):
    df[f"ventes_retard_{retard}"] = groupes["nombre_ventes"].shift(retard)

for fenetre in (3, 6, 12):
    df[f"ventes_moyenne_{fenetre}"] = groupes["nombre_ventes"].transform(
        lambda s: s.shift(1).rolling(fenetre, min_periods=2).mean()
    )

df["ventes_ecart_type_12"] = groupes["nombre_ventes"].transform(
    lambda s: s.shift(1).rolling(12, min_periods=3).std()
)
df["variation_mensuelle"] = df["ventes_retard_1"] / df["ventes_retard_2"] - 1
df["variation_annuelle"] = df["ventes_retard_1"] / df["ventes_retard_12"] - 1
df = df.replace([np.inf, -np.inf], np.nan)

## Contexte de marché

Part de la série dans le volume québécois du mois, et volume provincial,
tous deux décalés d'un mois.

In [6]:
df["total_quebec"] = df.groupby("periode")["nombre_ventes"].transform("sum")
df["part_quebec"] = df["nombre_ventes"] / df["total_quebec"]

par_serie = df.groupby(CLES_SERIE, group_keys=False)
df["part_quebec_retard_1"] = par_serie["part_quebec"].shift(1)
df["total_quebec_retard_1"] = par_serie["total_quebec"].shift(1)

## Cible et encodage

In [7]:
df["cible_ventes_suivantes"] = df.groupby(CLES_SERIE)["nombre_ventes"].shift(-1)

encodage_plage = pd.get_dummies(df["plage_prix"], prefix="plage", dtype=int)
encodage_region = pd.get_dummies(df["id_region"], prefix="region", dtype=int)
df = pd.concat([df, encodage_plage, encodage_region], axis=1)

In [8]:
COLONNES_EXPLICATIVES = [
    "annee",
    "mois_sin",
    "mois_cos",
    "trimestre",
    "ventes_retard_1",
    "ventes_retard_2",
    "ventes_retard_3",
    "ventes_retard_12",
    "ventes_moyenne_3",
    "ventes_moyenne_6",
    "ventes_moyenne_12",
    "ventes_ecart_type_12",
    "variation_mensuelle",
    "variation_annuelle",
    "part_quebec_retard_1",
    "total_quebec_retard_1",
    *encodage_plage.columns.tolist(),
    *encodage_region.columns.tolist(),
]

identifiants = ["periode", "id_region", "region", "code_plage_prix", "plage_prix"]
features = df[[*identifiants, *COLONNES_EXPLICATIVES, "cible_ventes_suivantes"]].dropna(
    subset=["cible_ventes_suivantes", "ventes_retard_12"]
)

print(f"{len(features)} observations, {len(COLONNES_EXPLICATIVES)} variables")
features.head()

2638 observations, 35 variables


,periode,id_region,region,code_plage_prix,plage_prix,annee,mois_sin,mois_cos,trimestre,ventes_retard_1,ventes_retard_2,ventes_retard_3,ventes_retard_12,ventes_moyenne_3,ventes_moyenne_6,ventes_moyenne_12,ventes_ecart_type_12,variation_mensuelle,variation_annuelle,part_quebec_retard_1,total_quebec_retard_1,plage_De 250 000 $ à 500 000 $,plage_Moins de 250 000 $,plage_Plus de 500 000 $,region_01,region_02,region_03,region_04,region_05,region_06,region_07,region_08,region_09,region_11,region_12,region_13,region_14,region_15,region_16,region_17,cible_ventes_suivantes
12,2022-01-01,01,Bas-Saint-Laurent,1,Moins de 250 000 $,2022,0.500000,8.660254e-01,1,416.0,443.0,452.0,240.0,437.000000,477.333333,471.416667,138.402219,-0.060948,0.733333,0.022378,18590.0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,310.0
13,2022-02-01,01,Bas-Saint-Laurent,1,Moins de 250 000 $,2022,0.866025,5.000000e-01,1,224.0,416.0,443.0,327.0,361.000000,407.166667,470.083333,140.889033,-0.461538,-0.314985,0.020047,11174.0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,355.0
14,2022-03-01,01,Bas-Saint-Laurent,1,Moins de 250 000 $,2022,1.000000,6.123234e-17,1,310.0,224.0,416.0,419.0,316.666667,394.333333,468.666667,142.534419,0.383929,-0.260143,0.022696,13659.0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,332.0
15,2022-04-01,01,Bas-Saint-Laurent,1,Moins de 250 000 $,2022,0.866025,-5.000000e-01,2,355.0,310.0,224.0,486.0,296.333333,366.666667,463.333333,145.723482,0.145161,-0.269547,0.022129,16042.0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,432.0
16,2022-05-01,01,Bas-Saint-Laurent,1,Moins de 250 000 $,2022,0.500000,-8.660254e-01,2,332.0,355.0,310.0,566.0,332.333333,346.666667,450.500000,150.256447,-0.064789,-0.413428,0.019172,17317.0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,536.0


In [9]:
taux_manquants = features[COLONNES_EXPLICATIVES].isna().mean().sort_values(ascending=False)
assert (taux_manquants < 0.5).all(), f"Variables trop incomplètes :\n{taux_manquants.head()}"
taux_manquants.head(10)

annee               0.0
mois_sin            0.0
mois_cos            0.0
trimestre           0.0
ventes_retard_1     0.0
ventes_retard_2     0.0
ventes_retard_3     0.0
ventes_retard_12    0.0
ventes_moyenne_3    0.0
ventes_moyenne_6    0.0
dtype: float64

In [10]:
FEATURES_CSV.parent.mkdir(parents=True, exist_ok=True)
features.to_csv(FEATURES_CSV, index=False, encoding="utf-8")
print(f"Écrit : {FEATURES_CSV}")

Écrit : C:\Users\Claude1\Documents\Projet d'analyse\Projet-marche-immobilier-quebec\data\processed\features.csv
